# Session 55B — Frozen Gate40 predictions

**UNAUTHORIZED / DO NOT RUN until separate owner Approval A (metric and runtime) and Approval B (held-out inference).** Run All exactly once after both approvals. This notebook consumes **free Kaggle GPU and Hugging Face download quota**; it makes no paid API calls. It generates only the fixed 40 Mind the Query predictions, never the remaining 260. Gold queries are absent from the attached packet.

Expected outputs: `s55_qwen_mtq_gate40.jsonl`, `s55_mistral_mtq_gate40.jsonl`, logs, environment and input-hash manifests, an execution-source export, and one ZIP. If any gate fails, the ZIP is marked incomplete; do not treat it as a scientific result.


In [ ]:
"""Standard-library setup/validation used verbatim in the self-contained Gate40 notebook."""
from pathlib import Path
import hashlib
import importlib.util
import json
import math
import os
import shutil
import stat
import subprocess
import sys
import zipfile

HASHES={
 "kaggle/run_inference.py":"8dbdb19653c84246898572b3dc79f02b0a9c8a00967a466526a5be36cb781bbf",
 "kaggle/prompt_template.txt":"13c431f01493eeeb52f9869257b182d0ac557c624dd5304e8fe435ba91b22f6e",
 "data/packet_mtq_gate40.jsonl":"836c3ba458c07dc32cad8dba3b19f9cfd3370713a0d0eea5807c71b8d867d219",
 "data/packet_mtq_300.jsonl":"a620130fa568f5e6293c38d320647a7ec063fc643db54ad110b21ccb4d130596",
 "data/packet_mtq_remaining260.jsonl":"e74497253c427aecf4708ab087874b4b3e7d32eba6abac0e9ff98c99dc635664",
 "data/packet_mtq_manifest.json":"5499c7188b039cb146f53b2c7598410c430f9b2904e1ba2c0235ea69d8372f9c",
}
MODELS={"qwen":("Qwen/Qwen2.5-Coder-7B-Instruct","c03e6d358207e414f1eca0bb1891e29f1db0e242"),
        "mistral":("mistralai/Mistral-7B-Instruct-v0.3","c170c708c41dac9275d15a8fff4eca08d52bab71")}
VERSIONS={"python":"3.13.16","torch":"2.11.0+cu130","cuda_runtime":"13.0","transformers":"5.20.2","bitsandbytes":"0.50.2","accelerate":"1.2.1"}

def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def safe_zip(path):
    with zipfile.ZipFile(path) as z:
        names=z.namelist()
        if len(names)!=len(set(names)) or sum(x.file_size for x in z.infolist())>100_000_000:
            raise ValueError("duplicate or oversized ZIP")
        for x in z.infolist():
            p=Path(x.filename)
            mode=x.external_attr>>16
            if p.is_absolute() or ".." in p.parts or x.filename.startswith("\\") or (mode and not (stat.S_ISREG(mode) or stat.S_ISDIR(mode))):
                raise ValueError("unsafe ZIP member")
        if z.testzip() is not None: raise ValueError("ZIP CRC")
        if not set(HASHES).issubset(set(names)): raise ValueError("incomplete ZIP")

def discover(input_root):
    root=Path(input_root)
    candidates=[]
    for p in root.rglob("run_inference.py"):
        if p.parent.name=="kaggle" and all((p.parent.parent/name).is_file() for name in HASHES):
            candidates.append(("directory",p.parent.parent))
    for p in root.rglob("kaggle_package_mtq.zip"):
        safe_zip(p)
        candidates.append(("zip",p))
    # An attached dataset can expose both the extracted directory and its ZIP.
    # Prefer the directory only if exactly one candidate of that kind exists.
    directories=[c for c in candidates if c[0]=="directory"]
    chosen=directories if directories else candidates
    if len(chosen)!=1: raise ValueError(f"expected unique attached package, found {len(chosen)}")
    return chosen[0]

def prepare(input_root, work):
    kind,source=discover(input_root)
    work=Path(work)
    work.mkdir(parents=True,exist_ok=False)
    if kind=="directory":
        for name in HASHES:
            dst=work/name;dst.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(source/name,dst)
    else:
        with zipfile.ZipFile(source) as z:
            for name in HASHES:
                dst=work/name;dst.parent.mkdir(parents=True,exist_ok=True)
                with z.open(name) as inp,dst.open("xb") as out: shutil.copyfileobj(inp,out)
    actual={name:sha(work/name) for name in HASHES}
    for name,expected in HASHES.items():
        if expected is not None and actual[name]!=expected: raise ValueError("frozen hash mismatch: "+name)
    manifest=json.loads((work/"data/packet_mtq_manifest.json").read_text())
    if manifest["packet_sha256"]!=HASHES["data/packet_mtq_300.jsonl"] or manifest["selected"]!=300:
        raise ValueError("manifest content mismatch")
    packets=[[json.loads(line) for line in (work/name).read_text().splitlines()] for name in (
      "data/packet_mtq_300.jsonl","data/packet_mtq_gate40.jsonl","data/packet_mtq_remaining260.jsonl")]
    ids=[[r["question_id"] for r in p] for p in packets]
    if [len(p) for p in packets]!=[300,40,260] or ids[0]!=ids[1]+ids[2] or any(len(set(seq))!=len(seq) for seq in ids):
        raise ValueError("packet membership")
    for row in packets[0]:
        if set(row)!={"question_id","benchmark","graph","question","schema"} or row["benchmark"]!="mind_the_query":
            raise ValueError("packet schema")
    return {"source_kind":kind,"source_path":str(source),"source_sha256":sha(source) if kind=="zip" else None,
            "file_sha256":actual,"gate_ids":ids[1]}

def validate_output(path, model, gate_ids, packet_sha, prompt_sha, runner_path):
    spec=importlib.util.spec_from_file_location("frozen_runner",runner_path)
    runner=importlib.util.module_from_spec(spec);spec.loader.exec_module(runner)
    rows=[json.loads(x) for x in Path(path).read_text().splitlines()]
    if len(rows)!=40 or [r.get("question_id") for r in rows]!=gate_ids: raise ValueError("missing/duplicate/order rows")
    counts={}
    for r in rows:
        if set(r)!={"question_id","status","raw_response","extracted_query","error","seconds","metadata"}:
            raise ValueError("row schema")
        m=r["metadata"]
        if not isinstance(r["seconds"],(int,float)) or not math.isfinite(r["seconds"]) or r["seconds"]<0:
            raise ValueError("invalid runtime duration")
        if (m.get("model"),m.get("revision"))!=MODELS[model] or m.get("packet_sha256")!=packet_sha or m.get("prompt_sha256")!=prompt_sha or m.get("smoke") is not False:
            raise ValueError("metadata mismatch")
        if any(m.get(k)!=VERSIONS[k] for k in ("python","torch","transformers")) or m.get("gpu")!="Tesla T4":
            raise ValueError("per-row runtime drift")
        if any(m.get(k)!=v for k,v in {"seed":55,"do_sample":False,"temperature":0,"max_prompt_tokens":4096,"max_new_tokens":256,"quantization":"NF4 4-bit double-quant float16"}.items()):
            raise ValueError("decoding drift")
        if r["status"]=="ok":
            if not r["extracted_query"] or r["extracted_query"]!=runner.extract(r["raw_response"]) or r["error"] is not None:
                raise ValueError("extraction")
        elif r["status"] in {"non_extractable","prompt_too_long","generation_error"}:
            if r["extracted_query"] is not None:raise ValueError("failure query")
            if r["status"]=="non_extractable" and (r["error"] is not None or runner.extract(r["raw_response"]) is not None):raise ValueError("nonextractable mismatch")
            if r["status"]=="prompt_too_long" and (r["raw_response"]!="" or r["error"] is not None):raise ValueError("length failure mismatch")
            if r["status"]=="generation_error" and not r["error"]:raise ValueError("generation error missing")
        else: raise ValueError("unknown status")
        counts[r["status"]]=counts.get(r["status"],0)+1
    return {"model":model,"count":40,"status_counts":counts,"sha256":sha(path)}

def package(run, files, target):
    run=Path(run);target=Path(target)
    if target.exists(): raise FileExistsError(target)
    if any(not (run/name).is_file() for name in files):raise ValueError("missing provenance file")
    with zipfile.ZipFile(target,"x",compression=zipfile.ZIP_DEFLATED) as z:
        for name in files:z.write(run/name,arcname=name)
    with zipfile.ZipFile(target) as z:
        if z.testzip() is not None or set(z.namelist())!=set(files):raise ValueError("bad output ZIP")
    return sha(target)


In [ ]:
from pathlib import Path
import datetime, importlib.metadata as md, json, os, platform, subprocess, sys, tempfile, traceback
BASE=Path('/kaggle/working')
RUN=Path(tempfile.mkdtemp(prefix='s55_gate40_',dir=BASE))
WORK=RUN/'input_copy'
INPUT=prepare('/kaggle/input',WORK)
(RUN/'input_hashes.json').write_text(json.dumps(INPUT,indent=2)+'\n')
print('Input verified:',INPUT['source_kind'], 'gate IDs:',len(INPUT['gate_ids']))


In [ ]:
# Fail closed on runtime drift; package evidence before the final assertion.
ENV={}; ERRORS=[]
try:
 import torch, transformers, bitsandbytes, accelerate
 ENV={'python':platform.python_version(),'torch':torch.__version__,
      'cuda_runtime':str(torch.version.cuda),'transformers':md.version('transformers'),
      'bitsandbytes':md.version('bitsandbytes'),'accelerate':md.version('accelerate'),
      'gpu_count':torch.cuda.device_count(),
      'gpu_names':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
 for k,v in VERSIONS.items():
  if ENV[k]!=v:raise RuntimeError(f'Runtime drift {k}: {ENV[k]} != {v}')
 if not torch.cuda.is_available() or ENV['gpu_count']<1 or not all(n=='Tesla T4' for n in ENV['gpu_names']):
  raise RuntimeError('T4 CUDA GPU unavailable')
 layer=bitsandbytes.nn.Linear4bit(16,8,compute_dtype=torch.float16,compress_statistics=True,quant_type='nf4').to('cuda:0')
 x=torch.ones((2,16),device='cuda:0',dtype=torch.float16)
 y=layer(x);torch.cuda.synchronize()
 if not y.is_cuda or tuple(y.shape)!=(2,8):raise RuntimeError('NF4 CUDA forward failed')
 ENV['nf4_cuda_test']='PASS'
except Exception as exc:
 ERRORS.append('runtime_preflight:'+type(exc).__name__+':'+str(exc))
 ENV['nf4_cuda_test']='FAIL'
(RUN/'environment.json').write_text(json.dumps(ENV,indent=2)+'\n')
try:
 probe=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],capture_output=True,text=True)
except FileNotFoundError as exc:
 probe=subprocess.CompletedProcess(['nvidia-smi'],-1,'',str(exc))
(RUN/'nvidia_smi.log').write_text('exit='+str(probe.returncode)+'\n'+probe.stdout+'\n'+probe.stderr)
snapshot=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True)
(RUN/'pip_freeze.log').write_text('exit='+str(snapshot.returncode)+'\n'+snapshot.stdout+'\n'+snapshot.stderr)
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
(RUN/'pip_check.log').write_text('exit='+str(check.returncode)+'\n'+check.stdout+'\n'+check.stderr)
if probe.returncode:ERRORS.append('nvidia_smi_failed')
print('Runtime preflight:',ENV.get('nf4_cuda_test'),'diagnostics:',ERRORS)


In [ ]:
# One attempt per model, sequential. Never --smoke; never retry content.
RESULTS={}
if not ERRORS:
 for key in ('qwen','mistral'):
  path=RUN/f's55_{key}_mtq_gate40.jsonl'
  command=[sys.executable,str(WORK/'kaggle/run_inference.py'),'--model',key,
           '--packet',str(WORK/'data/packet_mtq_gate40.jsonl'),'--output',str(path)]
  try:
   process=subprocess.run(command,capture_output=True,text=True,timeout=7200)
   (RUN/f'{key}_run.stdout.log').write_text(process.stdout)
   (RUN/f'{key}_run.stderr.log').write_text(process.stderr)
   RESULTS[key]={'command':command,'exit_code':process.returncode,'output_exists':path.exists()}
   if process.returncode:
    ERRORS.append(key+'_inference_exit_'+str(process.returncode))
    break
   try:
    RESULTS[key]['validation']=validate_output(path,key,INPUT['gate_ids'],HASHES['data/packet_mtq_gate40.jsonl'],HASHES['kaggle/prompt_template.txt'],WORK/'kaggle/run_inference.py')
   except Exception as exc:
    ERRORS.append(key+'_output_validation_'+type(exc).__name__+':'+str(exc))
    break
  except subprocess.TimeoutExpired as exc:
   (RUN/f'{key}_run.stdout.log').write_text(str(exc.stdout or ''))
   (RUN/f'{key}_run.stderr.log').write_text(str(exc.stderr or ''))
   RESULTS[key]={'command':command,'exit_code':'TIMEOUT','output_exists':path.exists()}
   ERRORS.append(key+'_timeout_7200s')
   break
  except Exception as exc:
   ERRORS.append(key+'_launch_'+type(exc).__name__+':'+str(exc))
   break
else:
 print('No model launched: preflight failed')


In [ ]:
# Package complete or partial evidence; incomplete runs are marked FAIL.
source='\n\n# ==== EXECUTED CELL ====\n\n'.join(get_ipython().history_manager.input_hist_raw[-5:])
(RUN/'notebook_executed_source.py').write_text(source)
MANIFEST={'phase':'gate40','complete':not ERRORS and set(RESULTS)=={'qwen','mistral'} and all('validation' in x for x in RESULTS.values()),
          'errors':ERRORS,'results':RESULTS,'input':INPUT,'environment':ENV,
          'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'notebook_executed_source_sha256':sha(RUN/'notebook_executed_source.py')}
(RUN/'execution_manifest.json').write_text(json.dumps(MANIFEST,indent=2)+'\n')
required=['input_hashes.json','environment.json','nvidia_smi.log','pip_freeze.log','pip_check.log',
          'execution_manifest.json','notebook_executed_source.py']
for key in RESULTS:
 for name in (f'{key}_run.stdout.log',f'{key}_run.stderr.log',f's55_{key}_mtq_gate40.jsonl'):
  if (RUN/name).exists():required.append(name)
(RUN/'SHA256SUMS').write_text(''.join(f'{sha(RUN/name)}  {name}\n' for name in required))
required.append('SHA256SUMS')
archive=RUN.with_name(RUN.name+'_artifacts.zip')
archive_sha=package(RUN,required,archive)
print('Archive:',archive,'SHA256:',archive_sha,'complete:',MANIFEST['complete'])
if not MANIFEST['complete']:raise RuntimeError('INCOMPLETE Gate40 evidence; inspect archive and do not score')
